In [ ]:
# Cell 1
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.evaluate import evaluate_cv
from src.models.predict import load_model
from src.utils import config

sns.set_style('whitegrid')

# Cell 2
bundle = load_model(config.BEST_MODEL_PATH)
train = pd.read_csv(config.FEATURES_TRAIN)
feats = bundle['features']
X, y = train[feats], train['TARGET']

oof, auc = evaluate_cv(bundle['model'], X, y)
train = train.assign(score=oof)
print('CV AUC:', round(auc, 5))

# Cell 3 — Score distribution by class
plt.figure(figsize=(8, 4))
for label, grp in train.groupby('TARGET'):
    sns.kdeplot(grp['score'], label=f'TARGET={label}', fill=True, alpha=0.3)
plt.title('Predicted score distribution by true class')
plt.xlabel('P(default)')
plt.legend()
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'score_distribution.png', dpi=120)
plt.show()

# Cell 4 — Worst false negatives
fn = train[train['TARGET'] == 1].nsmallest(10, 'score')
cols = ['uid', 'score', 'is_cash_loan', 'has_accounts', 'acc_n_accounts',
        'pmt_max_dpd', 'acc_open_ratio', 'enq_n', 'enq_days_since_last']
fn[[c for c in cols if c in fn.columns]]

# Cell 5 — Worst false positives
fp = train[train['TARGET'] == 0].nlargest(10, 'score')
fp[[c for c in cols if c in fp.columns]]

# Cell 6 — Error rate by segment
train['pred'] = (train['score'] >= 0.5).astype(int)
train['correct'] = (train['pred'] == train['TARGET']).astype(int)

print('Accuracy by has_accounts:')
print(train.groupby('has_accounts')['correct'].mean())

print('\nAccuracy by contract type:')
print(train.groupby('is_cash_loan')['correct'].mean())

# Cell 7 — Calibration by decile
train['decile'] = pd.qcut(train['score'], 10, labels=False, duplicates='drop')
calib = train.groupby('decile').agg(
    mean_score=('score', 'mean'),
    actual_rate=('TARGET', 'mean'),
    n=('TARGET', 'size'),
)
calib

# Cell 8
plt.figure(figsize=(6, 5))
plt.plot(calib['mean_score'], calib['actual_rate'], 'o-', label='model')
plt.plot([0, calib['mean_score'].max()], [0, calib['mean_score'].max()],
         'k--', alpha=0.4, label='perfect calibration')
plt.xlabel('Mean predicted score')
plt.ylabel('Actual default rate')
plt.title('Calibration by score decile')
plt.legend()
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'calibration.png', dpi=120)
plt.show()